# YOLO11s-Seg Floorplan Door & Window Segmentation Pipeline
### End-to-End Master Orchestrator: Dataset Synthesis, Model Training, ONNX Benchmarking, and Real-world Evaluation

This master notebook integrates both Phase 1 (Dataset Generation & Domain Randomization) and Phase 2 (Model Training & Multi-format Export Registry) into a single continuous pipeline. Checkpoints and versioned models are automatically synchronized directly with Google Drive.

## 1. Environment Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies for synthesis, training, ONNX optimization, and benchmarking
!pip install -q cairosvg svgpathtools bs4 lxml numpy opencv-python pyyaml kagglehub ultralytics onnx onnxslim onnxruntime pandas

import os
import sys
import shutil
import hashlib
import random
import re
import json
import glob
import cv2
import numpy as np
import torch
from pathlib import Path

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU runtime! Go to Runtime > Change runtime type and select GPU.")

## 2. Reconstruct Source Vector Math Codebase
We clone or assume the codebase from GitHub to load transform resolvers, coordinate mappers, and door anatomy extractors.

In [ ]:
repo_path = "/content/data_annotation"
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    
print("Cloning processing scripts from Git...")
!git clone https://github.com/for-real-afk/data_annotation.git {repo_path}

sys.path.append(repo_path)
sys.path.append(os.path.join(repo_path, "src"))
print("Source path configured.")

## 3. Phase 1: High-Fidelity Dataset Generation & Provenance Metadata

In [ ]:
import kagglehub
from bs4 import BeautifulSoup

# Re-import upgraded submodules locally
from svg_parser import get_svg_dimensions, collect_direct_geometry, parse_points
from transform_resolver import get_global_transform
from coordinate_mapper import CoordinateMapper
from door_anatomy_extractor import extract_door_info_cad
from door_extractor import is_furniture

# Dataset directories setup
dataset_root = "/content/yolo_dataset"
metadata_dir = os.path.join(dataset_root, "metadata")
splits = ["train", "val", "test"]

for split in splits:
    os.makedirs(os.path.join(dataset_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, "labels", split), exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)

# Window extraction function
def extract_windows_svg(soup, mapper, source_svg_file):
    windows = []
    window_idx = 0
    for tag in soup.find_all(['rect', 'polygon', 'path']):
        parent_id = str(tag.parent.get('id', '')).lower() if tag.parent else ""
        parent_class = str(tag.parent.get('class', '')).lower() if tag.parent else ""
        tag_id = str(tag.get('id', '')).lower()
        tag_class = str(tag.get('class', '')).lower()
        
        is_window = any(k in parent_id or k in parent_class or k in tag_id or k in tag_class for k in ['window', 'glass'])
        if not is_window: continue
        
        m_glob = get_global_transform(tag)
        pts = []
        if tag.name == 'path':
            d = tag.get('d', '')
            try:
                from svgpathtools import parse_path
                path_obj = parse_path(d)
                for segment in path_obj:
                    for t in np.linspace(0, 1, 5):
                        pt = segment.point(t)
                        pt_glob = m_glob @ np.array([pt.real, pt.imag, 1.0])
                        pts.append((pt_glob[0], pt_glob[1]))
            except: pass
        elif tag.name == 'polygon':
            coords = parse_points(tag.get('points', ''))
            for pt in coords:
                pt_glob = m_glob @ np.array([pt[0], pt[1], 1.0])
                pts.append((pt_glob[0], pt_glob[1]))
        elif tag.name == 'rect':
            try:
                rx = float(tag.get('x', 0))
                ry = float(tag.get('y', 0))
                rw = float(tag.get('width', 0))
                rh = float(tag.get('height', 0))
                coords = [(rx, ry), (rx+rw, ry), (rx+rw, ry+rh), (rx, ry+rh)]
                for pt in coords:
                    pt_glob = m_glob @ np.array([pt[0], pt[1], 1.0])
                    pts.append((pt_glob[0], pt_glob[1]))
            except: pass
            
        if len(pts) >= 3:
            pts_png = [mapper.svg_to_image(pt[0], pt[1]) for pt in pts]
            xs = [p[0] for p in pts_png]
            ys = [p[1] for p in pts_png]
            bbox = [min(xs), min(ys), max(xs), max(ys)]
            width_px = float(max(xs) - min(xs))
            
            parent_group = tag.parent.name if tag.parent else "svg"
            parent_id_str = tag.parent.get("id", "") if tag.parent else ""
            source_svg_group = f"{parent_group}#{parent_id_str}" if parent_id_str else parent_group
            
            windows.append({
                "window_id": f"w_{window_idx}",
                "bbox": bbox,
                "polygon": pts_png,
                "width_px": width_px,
                "scale_available": False,
                "source_class": "window",
                "source_svg_group": source_svg_group,
                "source_svg_file": source_svg_file
            })
            window_idx += 1
    return windows

# Download CubiCasa5K dataset
print("Downloading CubiCasa5K dataset...")
raw_dataset_path = kagglehub.dataset_download("qmarva/cubicasa5k")
svg_files = glob.glob(os.path.join(raw_dataset_path, "**", "model.svg"), recursive=True)
svg_files.extend(glob.glob(os.path.join(raw_dataset_path, "**", "model1.svg"), recursive=True))
svg_files = list(set(svg_files))
print(f"Found {len(svg_files)} plan SVGs in cache.")

random.seed(42)
random.shuffle(svg_files)

# 80% Train, 10% Val, 10% Test split
n_total = len(svg_files)
idx_train = int(n_total * 0.8)
idx_val = int(n_total * 0.9)

split_svgs = {
    "train": svg_files[:idx_train],
    "val": svg_files[idx_train:idx_val],
    "test": svg_files[idx_val:]
}

total_doors_count = 0
total_windows_count = 0
total_images_count = 0
processed_counts = {"train": 0, "val": 0, "test": 0}

limit = 5000
for split, files in split_svgs.items():
    print(f"Rendering and parsing {split} split...")
    for svg_path in files:
        if sum(processed_counts.values()) >= limit:
            break
            
        plan_name = os.path.basename(os.path.dirname(svg_path)) + "_" + os.path.basename(svg_path).replace(".svg", "")
        img_dst = os.path.join(dataset_root, "images", split, f"{plan_name}.png")
        lbl_dst = os.path.join(dataset_root, "labels", split, f"{plan_name}.txt")
        meta_dst = os.path.join(metadata_dir, f"{plan_name}.json")
        
        try:
            with open(svg_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f.read(), 'xml')
            svg_w, svg_h = get_svg_dimensions(soup)
            
            M_scale = np.array([
                [1024.0 / svg_w, 0, 0],
                [0, 1024.0 / svg_h, 0],
                [0, 0, 1]
            ])
            mapper = CoordinateMapper(M_scale)
            
            doors = []
            door_idx = 0
            for tag in soup.find_all():
                tag_id = str(tag.get("id", ""))
                tag_classes = tag.get("class", [])
                tag_classes_str = " ".join(tag_classes) if isinstance(tag_classes, list) else str(tag_classes)
                tag_text = (tag_id + " " + tag_classes_str).lower()
                
                if not re.search(r"\bdoor\b", tag_text): continue
                if is_furniture(tag): continue
                if any(p in [t for t in soup.find_all() if re.search(r"\bdoor\b", (str(t.get("id","")) + " " + str(t.get("class",""))).lower())] for p in tag.parents): continue
                
                door_res = extract_door_info_cad(tag, mapper, source_svg_file=os.path.basename(svg_path), door_idx=door_idx)
                if door_res:
                    if isinstance(door_res, list):
                        doors.extend(door_res)
                        door_idx += len(door_res)
                    else:
                        doors.append(door_res)
                        door_idx += 1
            
            windows = extract_windows_svg(soup, mapper, os.path.basename(svg_path))
            if not doors and not windows:
                continue
                
            plan_metadata = {
                "svg": os.path.basename(svg_path),
                "image_size": [1024, 1024],
                "doors": doors,
                "windows": windows
            }
            with open(meta_dst, "w") as mf:
                json.dump(plan_metadata, mf, indent=2)
                
            cairosvg.svg2png(url=svg_path, write_to=img_dst, output_width=1024, output_height=1024)
            
            with open(lbl_dst, "w") as lf:
                for door in doors:
                    poly_pts = door["polygon"]
                    if len(poly_pts) >= 3:
                        coords_str = " ".join([f"{max(0.0, min(1.0, pt[0]/1024.0)):.6f} {max(0.0, min(1.0, pt[1]/1024.0)):.6f}" for pt in poly_pts])
                        lf.write(f"0 {coords_str}\n")
                for win in windows:
                    poly_pts = win["polygon"]
                    if len(poly_pts) >= 3:
                        coords_str = " ".join([f"{max(0.0, min(1.0, pt[0]/1024.0)):.6f} {max(0.0, min(1.0, pt[1]/1024.0)):.6f}" for pt in poly_pts])
                        lf.write(f"1 {coords_str}\n")
            
            total_doors_count += len(doors)
            total_windows_count += len(windows)
            total_images_count += 1
            processed_counts[split] += 1
        except Exception as e:
            pass

print(f"Dataset generated. Images: {total_images_count} | Doors: {total_doors_count} | Windows: {total_windows_count}")

# Create dataset.yaml file locally
dataset_yaml_content = f"""path: {dataset_root}
train: images/train
val: images/val
test: images/test

names:
  0: door
  1: window
"""
with open(os.path.join(dataset_root, "dataset.yaml"), "w") as f:
    f.write(dataset_yaml_content)

## 4. Domain Randomization & QC QA Controls

In [ ]:
# Domain Randomization
train_img_paths = glob.glob(os.path.join(dataset_root, "images", "train", "*.png"))
random.shuffle(train_img_paths)
for img_path in train_img_paths[:int(len(train_img_paths) * 0.3)]:
    img = cv2.imread(img_path)
    if img is None: continue
    effect = random.choice(["blur", "noise", "contrast"])
    if effect == "blur":
        img = cv2.GaussianBlur(img, (5, 5), 0)
    elif effect == "noise":
        ret, enc = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), random.randint(20, 50)])
        img = cv2.imdecode(enc, 1)
    elif effect == "contrast":
        img = cv2.convertScaleAbs(img, alpha=random.uniform(0.7, 1.3), beta=random.randint(-30, 30))
    cv2.imwrite(img_path, img)

# QA Dashboards
qa_dir = os.path.join(dataset_root, "qa_dataset")
os.makedirs(qa_dir, exist_ok=True)
all_img_paths = glob.glob(os.path.join(dataset_root, "images", "*", "*.png"))
random.shuffle(all_img_paths)
for path in all_img_paths[:500]:
    img = cv2.imread(path)
    basename = os.path.basename(path)
    plan_name = os.path.splitext(basename)[0]
    
    # find label
    lbl = None
    for s in splits:
        p = os.path.join(dataset_root, "labels", s, f"{plan_name}.txt")
        if os.path.exists(p):
            lbl = p
            break
    if lbl:
        with open(lbl, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                cls = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                pts = []
                for j in range(0, len(coords), 2):
                    pts.append([int(coords[j]*1024), int(coords[j+1]*1024)])
                pts = np.array(pts, np.int32).reshape((-1, 1, 2))
                color = (0, 255, 0) if cls == 0 else (255, 0, 0)
                cv2.polylines(img, [pts], True, color, 2)
        cv2.imwrite(os.path.join(qa_dir, f"qa_{basename}"), img)
print("QC Dashboard overlays generated.")

## 5. Dataset Zip Packaging & SHA-256 Manifest Fingerprint

In [ ]:
zip_dest = "/content/dataset_export.zip"
if os.path.exists(zip_dest): os.remove(zip_dest)
shutil.make_archive("/content/dataset_export", "zip", dataset_root)

# Compute SHA-256
sha256 = hashlib.sha256()
with open(zip_dest, "rb") as f:
    while chunk := f.read(8192):
        sha256.update(chunk)
dataset_hash = sha256.hexdigest()
print(f"SHA-256 Fingerprint: {dataset_hash}")

manifest = {
    "dataset_version": "v1",
    "generator_commit": "master_pipeline",
    "date": "2026-06-23",
    "num_images": total_images_count,
    "num_doors": total_doors_count,
    "num_windows": total_windows_count,
    "num_train": processed_counts["train"],
    "num_val": processed_counts["val"],
    "num_test": processed_counts["test"],
    "image_size": 1024,
    "augmentation_profile": "v1",
    "dataset_hash": dataset_hash
}

manifest_path = os.path.join(dataset_root, "dataset_manifest.json")
with open(manifest_path, "w") as mf:
    json.dump(manifest, mf, indent=2)

# Re-archive to include manifest
shutil.make_archive("/content/dataset_export", "zip", dataset_root)

drive_project_dir = "/content/drive/MyDrive/BOM_Project"
os.makedirs(drive_project_dir, exist_ok=True)
shutil.copy2(zip_dest, os.path.join(drive_project_dir, "dataset_export.zip"))
shutil.copy2(manifest_path, os.path.join(drive_project_dir, "dataset_manifest.json"))
print("Dataset exported and synced to Google Drive.")

## 6. Phase 2: YOLO Training Execution

In [ ]:
from ultralytics import YOLO
from pathlib import Path

drive_project_dir = "/content/drive/MyDrive/BOM_Project/models"
os.makedirs(drive_project_dir, exist_ok=True)
drive_checkpoint = os.path.join(drive_project_dir, "yolo11s_seg", "weights", "last.pt")
local_yaml_path = os.path.join(dataset_root, "dataset.yaml")

DEBUG = False
model_type = "yolo11n-seg.pt" if DEBUG else "yolo11s-seg.pt"
epochs = 10 if DEBUG else 100

# Check auto-resume
resume_mode = False
if Path(drive_checkpoint).exists():
    print(f"Checkpoint detected at {drive_checkpoint}. Resuming training...")
    model = YOLO(drive_checkpoint)
    resume_mode = drive_checkpoint
else:
    print(f"Starting fresh training with baseline {model_type}...")
    model = YOLO(model_type)

try:
    print("Starting YOLO training...")
    results = model.train(
        data=local_yaml_path,
        epochs=epochs,
        imgsz=1024,
        batch=16,
        device=0,
        cache=True,
        amp=True,
        workers=4,
        patience=20,
        project=drive_project_dir,
        name="yolo11s_seg",
        plots=True,
        save=True,
        save_period=5,
        optimizer="AdamW",
        cos_lr=True,
        resume=resume_mode
    )
except Exception as e:
    print(f"Memory limit hit. Retrying with batch=8 fallback...")
    if resume_mode:
        model = YOLO(drive_checkpoint)
    results = model.train(
        data=local_yaml_path,
        epochs=epochs,
        imgsz=1024,
        batch=8,
        device=0,
        cache=True,
        amp=True,
        workers=4,
        patience=20,
        project=drive_project_dir,
        name="yolo11s_seg",
        plots=True,
        save=True,
        save_period=5,
        optimizer="AdamW",
        cos_lr=True,
        resume=resume_mode
    )

run_dir = str(results.save_dir)
print("Training directory returned:", run_dir)
with open("/content/run_dir.txt", "w") as f:
    f.write(run_dir)

## 7. Audit Check: Find results.csv and best.pt

In [ ]:
from pathlib import Path

print("Searching for results.csv...")
for p in Path("/content").rglob("results.csv"):
    print(p)
    
print("\nSearching for best.pt...")
for p in Path("/content").rglob("best.pt"):
    print(p)

## 8. Model Export, Registry Compilation, and ONNX runtime benchmarks

In [ ]:
import pickle
import onnxruntime as ort
import time

if os.path.exists("/content/run_dir.txt"):
    with open("/content/run_dir.txt", "r") as f:
        run_dir = f.read().strip()
else:
    run_dir = os.path.join(drive_project_dir, "yolo11s_seg")

local_registry_dir = "models/v1"
run_weights = os.path.join(run_dir, "weights", "best.pt")

def export_pipeline(weights_path, export_dir, img_size):
    print(f"Exporting weights {weights_path} to {export_dir}...")
    if not os.path.exists(weights_path): raise FileNotFoundError(weights_path)
    os.makedirs(export_dir, exist_ok=True)
    
    model = YOLO(weights_path)
    weights_dir = os.path.dirname(os.path.abspath(weights_path))
    
    onnx_path = model.export(format="onnx", imgsz=img_size, opset=12)
    ts_path = model.export(format="torchscript", imgsz=img_size)
    
    shutil.copy2(weights_path, os.path.join(export_dir, "best.pt"))
    if onnx_path and os.path.exists(onnx_path): shutil.copy2(onnx_path, os.path.join(export_dir, "best.onnx"))
    if ts_path and os.path.exists(ts_path): shutil.copy2(ts_path, os.path.join(export_dir, "best.torchscript"))
    
    metadata = {
        "classes": {0: "door", 1: "window"},
        "img_size": img_size,
        "model_type": "YOLO11s-Seg",
        "epochs": 100,
        "dataset": "SVG Floorplans",
        "weights": "best.pt"
    }
    with open(os.path.join(export_dir, "bom_model.pkl"), "wb") as f:
        pickle.dump(metadata, f)

export_pipeline(run_weights, local_registry_dir, 1024)

# ONNX Validate
best_onnx = os.path.join(local_registry_dir, "best.onnx")
try:
    session = ort.InferenceSession(best_onnx)
    input_name = session.get_inputs()[0].name
    output_names = [o.name for o in session.get_outputs()]
    dummy_input = np.random.randn(1, 3, 1024, 1024).astype(np.float32)
    outputs = session.run(output_names, {input_name: dummy_input})
    print("ONNX session loading: OK")
except Exception as e:
    print(f"ONNX check failed: {e}")

# Benchmarks
sample_image = np.random.randint(0, 255, (1024, 1024, 3), dtype=np.uint8)
pt_model = YOLO(os.path.join(local_registry_dir, "best.pt"))
# Warmup
for _ in range(5): _ = pt_model(sample_image, imgsz=1024, verbose=False)
t0 = time.perf_counter()
for _ in range(20): _ = pt_model(sample_image, imgsz=1024, verbose=False)
pt_lat = (time.perf_counter() - t0) / 20.0 * 1000

input_data = cv2.resize(sample_image, (1024, 1024)).transpose(2, 0, 1)
input_data = np.expand_dims(input_data, axis=0).astype(np.float32) / 255.0
for _ in range(5): _ = session.run(output_names, {input_name: input_data})
t0 = time.perf_counter()
for _ in range(20): _ = session.run(output_names, {input_name: input_data})
onnx_lat = (time.perf_counter() - t0) / 20.0 * 1000

benchmarks = {
    "pytorch": {"latency_ms": pt_lat, "fps": 1000.0/pt_lat},
    "onnx": {"latency_ms": onnx_lat, "fps": 1000.0/onnx_lat}
}
with open(os.path.join(local_registry_dir, "inference_benchmark.json"), "w") as f:
    json.dump(benchmarks, f, indent=2)

## 9. Real-World Evaluation, BOM count MAE, and Failure Taxonomy

In [ ]:
hard_examples_root = os.path.join(local_registry_dir, "hard_examples")
missed_doors_dir = os.path.join(hard_examples_root, "missed_doors")
false_doors_dir = os.path.join(hard_examples_root, "false_doors")
partial_masks_dir = os.path.join(hard_examples_root, "partial_masks")
window_confusions_dir = os.path.join(hard_examples_root, "window_confusions")

for p in [missed_doors_dir, false_doors_dir, partial_masks_dir, window_confusions_dir]:
    os.makedirs(p, exist_ok=True)

val_img_dir = None
val_lbl_dir = None
candidates = [
    ("/content/real_validation/images", "/content/real_validation/labels"),
    ("/content/data_annotation/good_ones_dataset/images/val", "/content/data_annotation/good_ones_dataset/labels/val"),
    ("/content/yolo_dataset/images/val", "/content/yolo_dataset/labels/val")
]
for c_img, c_lbl in candidates:
    if os.path.exists(c_img) and len(os.listdir(c_img)) > 0:
        val_img_dir = c_img
        val_lbl_dir = c_lbl
        break

def iou_boxes(b1, b2):
    xi1 = max(b1[0], b2[0])
    yi1 = max(b1[1], b2[1])
    xi2 = min(b1[2], b2[2])
    yi2 = min(b1[3], b2[3])
    inter = max(0.0, xi2 - xi1) * max(0.0, yi2 - yi1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 1e-5 else 0.0

if val_img_dir:
    eval_images = glob.glob(os.path.join(val_img_dir, "*"))
    eval_images = [e for e in eval_images if e.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    total_gt_doors = 0
    total_gt_windows = 0
    total_pred_doors = 0
    total_pred_windows = 0
    door_mae_sum = 0.0
    window_mae_sum = 0.0
    labeled_files_count = 0
    
    for path in eval_images:
        basename = os.path.basename(path)
        plan_name = os.path.splitext(basename)[0]
        img = cv2.imread(path)
        h_img, w_img = img.shape[:2]
        
        gt_boxes = []
        gt_lbl_path = os.path.join(val_lbl_dir, f"{plan_name}.txt") if val_lbl_dir else ""
        
        gt_doors = 0
        gt_windows = 0
        if os.path.exists(gt_lbl_path) and os.path.getsize(gt_lbl_path) > 0:
            with open(gt_lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts: continue
                    cls = int(parts[0])
                    coords = [float(x) for x in parts[1:]]
                    xs = coords[0::2]
                    ys = coords[1::2]
                    x1 = min(xs) * w_img
                    y1 = min(ys) * h_img
                    x2 = max(xs) * w_img
                    y2 = max(ys) * h_img
                    gt_boxes.append((cls, [x1, y1, x2, y2]))
                    if cls == 0: gt_doors += 1
                    elif cls == 1: gt_windows += 1
            total_gt_doors += gt_doors
            total_gt_windows += gt_windows
            labeled_files_count += 1
            
        res = pt_model.predict(source=path, imgsz=1024, conf=0.25, verbose=False)[0]
        pred_boxes = []
        pred_doors = 0
        pred_windows = 0
        
        if res.boxes is not None:
            for box in res.boxes:
                cls = int(box.cls[0].item())
                conf = float(box.conf[0].item())
                xyxy = box.xyxy[0].tolist()
                pred_boxes.append((cls, xyxy, conf))
                if cls == 0: pred_doors += 1
                elif cls == 1: pred_windows += 1
                    
        total_pred_doors += pred_doors
        total_pred_windows += pred_windows
        
        if os.path.exists(gt_lbl_path):
            door_mae_sum += abs(gt_doors - pred_doors)
            window_mae_sum += abs(gt_windows - pred_windows)
            
        # taxonomy
        matched_gt = set()
        matched_pred = set()
        for p_idx, (p_cls, p_box, p_conf) in enumerate(pred_boxes):
            best_iou = 0.0
            best_gt_idx = -1
            for g_idx, (g_cls, g_box) in enumerate(gt_boxes):
                if g_idx in matched_gt: continue
                iou = iou_boxes(p_box, g_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = g_idx
            if best_gt_idx != -1 and best_iou > 0.3:
                g_cls, g_box = gt_boxes[best_gt_idx]
                matched_gt.add(best_gt_idx)
                matched_pred.add(p_idx)
                if p_cls != g_cls:
                    cv2.imwrite(os.path.join(window_confusions_dir, f"conf_{basename}"), img)
                elif best_iou < 0.5:
                    cv2.imwrite(os.path.join(partial_masks_dir, f"partial_{basename}"), img)
        for g_idx, (g_cls, g_box) in enumerate(gt_boxes):
            if g_idx not in matched_gt and g_cls == 0:
                cv2.imwrite(os.path.join(missed_doors_dir, f"missed_{basename}"), img)
        for p_idx, (p_cls, p_box, p_conf) in enumerate(pred_boxes):
            if p_idx not in matched_pred and p_cls == 0:
                cv2.imwrite(os.path.join(false_doors_dir, f"false_{basename}"), img)
                
    door_mae = door_mae_sum / max(1, labeled_files_count)
    window_mae = window_mae_sum / max(1, labeled_files_count)
    
    bom_metrics = {
        "eval_dataset": os.path.basename(val_img_dir),
        "total_images": len(eval_images),
        "total_gt_doors": total_gt_doors,
        "total_pred_doors": total_pred_doors,
        "total_gt_windows": total_gt_windows,
        "total_pred_windows": total_pred_windows,
        "door_count_mae": door_mae,
        "window_count_mae": window_mae
    }
    with open(os.path.join(local_registry_dir, "bom_metrics.json"), "w") as f:
        json.dump(bom_metrics, f, indent=2)
        
    domain_gap = {
        "synthetic_vs_real_gap": {
            "door_mae": door_mae,
            "window_mae": window_mae,
            "door_count_ratio": float(total_pred_doors) / max(1, total_gt_doors),
            "window_count_ratio": float(total_pred_windows) / max(1, total_gt_windows),
            "domain_shift_index": float(abs(total_gt_doors - total_pred_doors) + abs(total_gt_windows - total_pred_windows)) / max(1, total_gt_doors + total_gt_windows)
        }
    }
    with open(os.path.join(local_registry_dir, "domain_gap_report.json"), "w") as f:
        json.dump(domain_gap, f, indent=2)
    print(f"Real validation MAE: Door={door_mae:.4f} | Window={window_mae:.4f}")
else:
    print("No validation data found.")

## 10. Registry Bundling and Backup Synchronization to Drive

In [ ]:
import pandas as pd
import shutil

drive_project_base = "/content/drive/MyDrive/BOM_Project"
csv_path = os.path.join(run_dir, "results.csv")
summary = {}
if os.path.exists(csv_path):
    try:
        df = pd.read_csv(csv_path)
        final_row = df.iloc[-1]
        summary = {
            "dataset": "SVG Synthetic Floorplans",
            "epochs": len(df),
            "precision": float(final_row.get('metrics/precision(B)', 0.0)),
            "recall": float(final_row.get('metrics/recall(B)', 0.0)),
            "map50_box": float(final_row.get('metrics/mAP50(B)', 0.0)),
            "map5095_box": float(final_row.get('metrics/mAP50-95(B)', 0.0)),
            "map50_mask": float(final_row.get('metrics/mAP50(M)', 0.0)),
            "map5095_mask": float(final_row.get('metrics/mAP50-95(M)', 0.0))
        }
    except: pass
    
with open(os.path.join(local_registry_dir, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

def package_artifacts(export_dir, training_run_dir, dataset_dir, zip_dest_dir, drive_backup_dir):
    os.makedirs(zip_dest_dir, exist_ok=True)
    metrics_files = ["results.csv", "PR_curve.png", "F1_curve.png", "confusion_matrix.png", "args.yaml"]
    for f_name in metrics_files:
        src = os.path.join(training_run_dir, f_name)
        dst = os.path.join(export_dir, f_name)
        if os.path.exists(src): shutil.copy2(src, dst)
        
    bundle_zip_path = os.path.join(zip_dest_dir, "bom_training_bundle.zip")
    if os.path.exists(bundle_zip_path): os.remove(bundle_zip_path)
    shutil.make_archive(os.path.join(zip_dest_dir, "bom_training_bundle"), "zip", export_dir)
    
    dataset_zip_path = os.path.join(zip_dest_dir, "dataset_export.zip")
    if os.path.exists(dataset_zip_path): os.remove(dataset_zip_path)
    if os.path.exists(dataset_dir):
        shutil.make_archive(os.path.join(zip_dest_dir, "dataset_export"), "zip", dataset_dir)
        
    if drive_backup_dir:
        try:
            version_dir = os.path.basename(os.path.normpath(export_dir))
            drive_version_dir = os.path.join(drive_backup_dir, "models", version_dir)
            shutil.copytree(export_dir, drive_version_dir, dirs_exist_ok=True)
            for zip_file in [bundle_zip_path, dataset_zip_path]:
                if os.path.exists(zip_file):
                    shutil.copy2(zip_file, os.path.join(drive_backup_dir, os.path.basename(zip_file)))
            print("Google Drive backup successful!")
        except Exception as e:
            print(f"Drive backup skipped/failed: {e}")

package_artifacts(local_registry_dir, run_dir, dataset_root, ".", drive_project_base)
print("Master pipeline complete! All artifacts synchronized on Google Drive.")